# 使用生成模型进行文本分类

## 1. FlAN-T5

In [5]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
from transformers import pipeline
from datasets import load_dataset
from tqdm import tqdm
from transformers.pipelines.pt_utils import KeyDataset

In [4]:
pipe = pipeline("text2text-generation", model="google/flan-t5-small", device="cuda:0")

Device set to use cuda:0


In [6]:

# 影评数据
data = load_dataset("rotten_tomatoes", cache_dir="./cache")
data

Using the latest cached version of the dataset since rotten_tomatoes couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at cache\rotten_tomatoes\default\0.0.0\aa13bc287fa6fcab6daf52f0dfb9994269ffea28 (last modified on Tue Jan 27 08:39:54 2026).


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

In [ ]:
prompt = "Is the following sentence positive or negative?"
# 每个文档前加上提示词，明确模型的输出是正面还是负面
data = data.map(lambda example: {"ts": prompt + example["text"]})
data

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'ts'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label', 'ts'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label', 'ts'],
        num_rows: 1066
    })
})

In [7]:
y_pred = []
for output in tqdm(pipe(KeyDataset(data["test"], "ts")), total=len(data["test"])):
    text = output[0]["generated_text"]
    if text != "negative" and text != "positive":
        print("predict unknown:", text)
    y_pred.append(0 if text == "negative" else 1)

100%|██████████| 1066/1066 [00:44<00:00, 24.14it/s]


In [1]:
from sklearn.metrics import classification_report

def evaluate_performance(y_true, y_pred):
    performance = classification_report(y_true, y_pred, target_names=["negative", "positive"])
    print(performance)

In [9]:
evaluate_performance(data["test"]["label"], y_pred)

              precision    recall  f1-score   support

    negative       0.83      0.84      0.83       533
    positive       0.84      0.83      0.83       533

    accuracy                           0.83      1066
   macro avg       0.83      0.83      0.83      1066
weighted avg       0.83      0.83      0.83      1066



# 2. ChatGPT

In [1]:
import openai
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
from datasets import load_dataset

client = openai.OpenAI(base_url="http://10.225.67.30:3000/v1", api_key="sk-numkdXZ3RqM1qN3DA7D413DaF8634e488572A6CdE9026520")

In [2]:
def chatgpt_generation(prompt, document, model):
    """ 基于提示词和输入文档生成输出 """
    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": prompt.replace("[Document]", document)
        }
    ]
    chat_completion = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0,
        n=1
    )
    return chat_completion.choices[0].message.content

In [3]:
prompt = """predict whether the following sentence is positive or negative movie review:
[Document]
If it is positive return 1 and if it is negative return 0. Do not give any other answers.
"""
document = "unpretenious, charming, quirky, orginal"
chatgpt_generation(prompt, document, model="Qwen3-Coder-30B")

'1'

In [4]:
from sklearn.metrics import classification_report
from tqdm import tqdm

def evaluate_performance(y_true, y_pred):
    performance = classification_report(y_true, y_pred, target_names=["negative", "positive"])
    print(performance)

In [5]:
# 影评数据
data = load_dataset("rotten_tomatoes", cache_dir="./cache")
data
predictions = [
    chatgpt_generation(prompt, doc, model="Qwen3-Coder-30B") for doc in tqdm(data["test"]["text"])
]

Using the latest cached version of the dataset since rotten_tomatoes couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at cache\rotten_tomatoes\default\0.0.0\aa13bc287fa6fcab6daf52f0dfb9994269ffea28 (last modified on Tue Jan 27 08:39:54 2026).
100%|██████████| 1066/1066 [01:35<00:00, 11.14it/s]


In [6]:
# pred[0] 防止有的预测结果后面还有其它的输出
y_pred = [int(pred[0]) for pred in predictions]
evaluate_performance(data["test"]["label"], y_pred)

              precision    recall  f1-score   support

    negative       0.88      0.95      0.92       533
    positive       0.95      0.87      0.91       533

    accuracy                           0.91      1066
   macro avg       0.91      0.91      0.91      1066
weighted avg       0.91      0.91      0.91      1066

